# Explainable Unsupervised Surface Defect Detection with Feature Selection
**Implementation of:** *An Efficient and Explainable Unsupervised Framework for Industrial Surface Defect Detection Using Multi-Scale Feature Learning*

This single notebook implements every component in Section III of the paper and runs the protocol in Section IV:

| Paper section | What this notebook does |
|---|---|
| III-B Patch features | Wide ResNet-50, stages 2 & 3, 3×3 neighbourhood averaging, upsample + concat |
| III-C Few-shot mode | Variance filter (quantile τv) → Laplacian score on heat-kernel kNN graph → top-*d* channels → greedy coreset (sparse random projection) |
| III-D Illumination mode | Photometric copies *T(x)* → Laplacian score on graph **S = S_knn + λ S_pair** → correlation filter (τc) |
| III-E Multi-scale mode | Separate memory bank per stage, SPEC ranking per stage, held-out normalisation, softmax-weighted fusion (temperature *T*) |
| III-F Explainability | Exact channel decomposition **A²(p) = Σ c_j(p)** + matched training patch |
| IV Protocol | Image AUROC, pixel AUROC, AU-PRO (FPR ≤ 0.3), #channels *d*, memory-bank MB, ms/image; Table I (few-shot), Table II (ablation under lighting shift) |

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better). Then run the cells top to bottom.
A quick run (3 categories) takes roughly 15–25 min on a T4; the full 15-category protocol takes a few hours — the results are saved after every category, so you can resume.

## 0. Setup

In [ ]:
!pip -q install kagglehub
import os, glob, time, math, random, json, warnings, gc
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from PIL import Image
from scipy import ndimage
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU — switch the runtime to GPU!")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(0)

## 1. Configuration
All hyper-parameters named in the paper live here (they are the values to report in *Implementation Details*, Section IV-D).

In [ ]:
CFG = dict(
    # ---------------- data ----------------
    dataset       = "mvtec_ad",          # "mvtec_ad" or "mvtec_ad2"
    data_root     = "/content/data",     # where the dataset will be put / looked for
    all_categories= ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut",
                     "pill","screw","tile","toothbrush","transistor","wood","zipper"],
    run_categories= ["bottle","carpet","screw"],   # quick run; set to CFG["all_categories"] for the full paper protocol
    img_size      = 256,                 # resize shorter side
    crop          = 224,                 # centre crop
    # ---------------- backbone (III-B) ----------------
    backbone      = "wide_resnet50_2",
    pretrained    = True,
    # ---------------- selection (III-C/D/E) ----------------
    var_quantile  = 0.10,   # τv : drop channels whose variance is in the lowest 10 %
    keep_frac     = 0.20,   # d = keep_frac · D  (D=1536 concat → d≈307; per level: 102 / 205)
    n_graph       = 2000,   # Ns : patches sampled to build the graph
    knn           = 10,     # kg : neighbours per node
    lam_pair      = 1.0,    # λ  : weight of photometric pair edges
    corr_thresh   = 0.90,   # τc : max |Pearson ρ| between kept channels
    temperature   = 0.05,   # T  : softmax temperature for level weights (Eq. 5)
    holdout_frac  = 0.10,   # fraction of normal patches held out to normalise level maps
    # ---------------- memory bank ----------------
    coreset_ratio = 0.10,   # r
    min_bank      = 1000,   # never shrink the bank below this many entries (matters for k=1)
    proj_dim      = 128,    # sparse random projection dim used only for coreset selection
    # ---------------- scoring ----------------
    smooth_sigma  = 4,
    # ---------------- protocol (IV) ----------------
    k_shots       = [1, 2, 4, 8, 16],
    seeds         = [0, 1, 2],
    ablation_k    = 16,     # fitting images for Table II (None = all training images)
    results_dir   = "/content/results",
)
os.makedirs(CFG["results_dir"], exist_ok=True)

# Photometric transform used at FIT time (III-D) ...
FIT_T  = dict(gamma=[(0.70, 1.40)], contrast=(0.80, 1.20), brightness=0.10, shade=(0.00, 0.30))
# ... and a DIFFERENT, harsher range used only to create the shifted TEST set (IV-A controlled variant)
TEST_T = dict(gamma=[(0.40, 0.60), (1.60, 2.20)], contrast=(0.55, 0.75), brightness=0.20, shade=(0.40, 0.60))

## 2. Get the dataset
Pick **one** option. Option A (Kaggle mirror, no sign-up needed for public datasets in most cases) is the easiest.
The cell finds the folder that contains `bottle/train/good` automatically, wherever it was unpacked.

* **Option A – Kaggle mirror** (default).
* **Option B – Google Drive**: put `mvtec_anomaly_detection.tar.xz` (from https://www.mvtec.com/company/research/datasets/mvtec-ad) or the extracted folder in your Drive and set `DRIVE_PATH`.
* **MVTec AD 2** requires registration on the MVTec website; download it to Drive, set `CFG["dataset"]="mvtec_ad2"` and use Option B.

In [ ]:
SOURCE     = "kaggle"     # "kaggle" or "drive"
DRIVE_PATH = "/content/drive/MyDrive/mvtec_anomaly_detection.tar.xz"   # file OR folder, used when SOURCE="drive"

def find_root(base, probe):
    for dirpath, dirnames, _ in os.walk(base):
        if os.path.isdir(os.path.join(dirpath, probe)):
            return dirpath
    return None

PROBE = "bottle/train/good" if CFG["dataset"] == "mvtec_ad" else "can/train/good"
DATA_DIR = find_root(CFG["data_root"], PROBE) if os.path.isdir(CFG["data_root"]) else None

if DATA_DIR is None and SOURCE == "kaggle":
    import kagglehub
    # If this asks for credentials: upload kaggle.json or set os.environ["KAGGLE_USERNAME"/"KAGGLE_KEY"]
    p = kagglehub.dataset_download("ipythonx/mvtec-ad")
    DATA_DIR = find_root(p, PROBE)
elif DATA_DIR is None and SOURCE == "drive":
    from google.colab import drive; drive.mount("/content/drive")
    if os.path.isdir(DRIVE_PATH):
        DATA_DIR = find_root(DRIVE_PATH, PROBE)
    else:
        os.makedirs(CFG["data_root"], exist_ok=True)
        print("Extracting archive (≈5 GB, a few minutes)…")
        !tar -xf "{DRIVE_PATH}" -C "{CFG['data_root']}"
        DATA_DIR = find_root(CFG["data_root"], PROBE)

assert DATA_DIR is not None, "Dataset not found — check SOURCE / DRIVE_PATH"
print("Dataset root:", DATA_DIR)
print("Categories found:", sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d, "train"))))

## 3. Data loading

In [ ]:
def _load_rgb(path):
    img = Image.open(path).convert("RGB")
    if CFG["dataset"] == "mvtec_ad2":          # large non-square images → square resize, no crop
        img = img.resize((CFG["crop"], CFG["crop"]), Image.BILINEAR)
    else:
        img = TF.center_crop(TF.resize(img, CFG["img_size"], interpolation=TF.InterpolationMode.BILINEAR), CFG["crop"])
    return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)        # uint8 (3,H,W)

def _load_mask(path):
    if path is None or not os.path.exists(path):
        return torch.zeros(CFG["crop"], CFG["crop"], dtype=torch.bool)
    m = Image.open(path).convert("L")
    if CFG["dataset"] == "mvtec_ad2":
        m = m.resize((CFG["crop"], CFG["crop"]), Image.NEAREST)
    else:
        m = TF.center_crop(TF.resize(m, CFG["img_size"], interpolation=TF.InterpolationMode.NEAREST), CFG["crop"])
    return torch.from_numpy(np.asarray(m) > 127)

def list_category(root, cat):
    exts = ("*.png", "*.jpg", "*.bmp")
    g = lambda d: sorted(sum([glob.glob(os.path.join(d, e)) for e in exts], []))
    train = g(os.path.join(root, cat, "train", "good"))
    test_dir = os.path.join(root, cat, "test" if CFG["dataset"] == "mvtec_ad" else "test_public")
    test = []
    for defect in sorted(os.listdir(test_dir)):
        if defect == "ground_truth" or not os.path.isdir(os.path.join(test_dir, defect)): continue
        for p in g(os.path.join(test_dir, defect)):
            stem = os.path.splitext(os.path.basename(p))[0]
            if defect == "good": mp = None
            elif CFG["dataset"] == "mvtec_ad": mp = os.path.join(root, cat, "ground_truth", defect, stem + "_mask.png")
            else: mp = os.path.join(test_dir, "ground_truth", defect, stem + "_mask.png")
            test.append((p, int(defect != "good"), mp, defect))
    return train, test

def load_category(root, cat):
    train, test = list_category(root, cat)
    X_train = torch.stack([_load_rgb(p) for p in tqdm(train, desc=f"{cat} train", leave=False)])
    X_test  = torch.stack([_load_rgb(p) for p, *_ in tqdm(test, desc=f"{cat} test", leave=False)])
    M_test  = torch.stack([_load_mask(mp) for _, _, mp, _ in test])
    y_test  = np.array([t[1] for t in test]); types = [t[3] for t in test]
    return dict(cat=cat, X_train=X_train, X_test=X_test, M_test=M_test, y_test=y_test, types=types)

_tr, _te = list_category(DATA_DIR, CFG["run_categories"][0])
print(f"{CFG['run_categories'][0]}: {len(_tr)} train / {len(_te)} test images")

## 4. Photometric transform *T* (Section III-D)
Gamma + contrast/brightness jitter + a smooth multiplicative shading field with random direction. Geometry is untouched, so patch *a* of *x* and patch *a′* of *T(x)* cover the same surface location.

In [ ]:
def photometric(x_uint8, rng, gamma, contrast, brightness, shade):
    x = x_uint8.float() / 255.0
    lo, hi = gamma[rng.integers(len(gamma))]
    x = x.clamp_min(1e-6) ** rng.uniform(lo, hi)
    m = x.mean()
    x = (x - m) * rng.uniform(*contrast) + m + rng.uniform(-brightness, brightness)
    H, W = x.shape[-2:]
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing="ij")
    th = rng.uniform(0, 2 * np.pi)
    ramp = (np.cos(th) * xx + np.sin(th) * yy) / np.sqrt(2)              # in [-1, 1]
    x = x * (1 + rng.uniform(*shade) * ramp)
    return (x.clamp(0, 1) * 255).round().to(torch.uint8)

def perturb_batch(X, params, seed):
    rng = np.random.default_rng(seed)
    return torch.stack([photometric(x, rng, **params) for x in X])

# preview
_x = _load_rgb(_tr[0])
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
ax[0].imshow(_x.permute(1, 2, 0)); ax[0].set_title("original")
for i in range(2):
    ax[1 + i].imshow(perturb_batch(_x[None], FIT_T, i)[0].permute(1, 2, 0)); ax[1 + i].set_title(f"fit-time T(x) #{i}")
    ax[3 + i].imshow(perturb_batch(_x[None], TEST_T, 10 + i)[0].permute(1, 2, 0)); ax[3 + i].set_title(f"test lighting shift #{i}")
for a in ax: a.axis("off")
plt.show()

## 5. Patch feature extraction (Section III-B)
Stages 2 and 3 of an ImageNet Wide ResNet-50, each position averaged over its 3×3 neighbourhood. Features are cached per category (float16 on CPU) so that every method and every few-shot draw reuses them.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
LEVELS = ("layer2", "layer3")

class Backbone(nn.Module):
    def __init__(self, name, pretrained=True):
        super().__init__()
        m = getattr(torchvision.models, name)(weights="IMAGENET1K_V1" if pretrained else None)
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool)
        self.layer1, self.layer2, self.layer3 = m.layer1, m.layer2, m.layer3
        self.eval()
    @torch.no_grad()
    def forward(self, x_uint8):
        x = (x_uint8.float() / 255.0 - MEAN.to(x_uint8.device)) / STD.to(x_uint8.device)
        f1 = self.layer1(self.stem(x)); f2 = self.layer2(f1); f3 = self.layer3(f2)
        pool = lambda f: F.avg_pool2d(f, 3, stride=1, padding=1)       # 3×3 local neighbourhood aggregation
        return {"layer2": pool(f2), "layer3": pool(f3)}

backbone = Backbone(CFG["backbone"], CFG["pretrained"]).to(DEVICE)

@torch.no_grad()
def extract(X, bs=32):
    out = {l: [] for l in LEVELS}; t0 = time.time()
    for i in range(0, len(X), bs):
        f = backbone(X[i:i + bs].to(DEVICE))
        for l in LEVELS: out[l].append(f[l].half().cpu())
    if DEVICE == "cuda": torch.cuda.synchronize()
    ms = (time.time() - t0) * 1000 / max(len(X), 1)
    return {l: torch.cat(v) for l, v in out.items()}, ms

def subset(feats, idx):
    return {l: v[idx] for l, v in feats.items()}

def concat_levels(feats):
    # III-B: upsample the coarser map bilinearly to the finer grid and concatenate → φ(p) ∈ R^D, D = 512 + 1024
    f2 = feats["layer2"].float(); f3 = feats["layer3"].float()
    f3 = F.interpolate(f3, size=f2.shape[-2:], mode="bilinear", align_corners=False)
    return torch.cat([f2, f3], 1)

def to_patches(fmap):                       # (n,C,h,w) → (n·h·w, C)
    return fmap.permute(0, 2, 3, 1).reshape(-1, fmap.shape[1])

_f, _ms = extract(_load_rgb(_tr[0])[None].repeat(2, 1, 1, 1))
print({l: tuple(v.shape) for l, v in _f.items()}, "| concat D =", concat_levels(_f).shape[1], f"| {_ms:.1f} ms/img")

## 6. Unsupervised feature selection (Sections III-C, III-D, III-E)
* **Variance filter** — Eq. (1), threshold τv taken as a quantile.
* **Heat-kernel kNN graph** — S_ab = exp(−‖φ_a−φ_b‖²/t), symmetrised, *t* = mean squared kNN distance.
* **Consistency graph** — S = S_knn + λ·S_pair, where S_pair links each patch to its photometric copy.
* **Laplacian score** — Eq. (2). **SPEC** (first ranking function, identity spectral function) — Eq. (4). Lower is better for both.
* **Correlation filter** — visit channels by ascending score, keep *j* only if |ρ_jk| ≤ τc for every kept *k*.

In [ ]:
EPS = 1e-12

def variance_filter(P, q):
    v = P.var(0, unbiased=False)                                   # Eq. (1)
    tau = torch.quantile(v.float(), q)
    return torch.nonzero(v > tau).squeeze(1)

def knn_heat_graph(X, k):
    d2 = torch.cdist(X, X).pow(2)
    d2.fill_diagonal_(float("inf"))
    k = min(k, len(X) - 1)
    val, idx = d2.topk(k, dim=1, largest=False)
    t = val.mean().clamp_min(EPS)
    S = torch.zeros_like(d2)
    S.scatter_(1, idx, torch.exp(-val / t))
    return torch.maximum(S, S.T)                                    # symmetric kNN graph

def laplacian_score(X, S):                                          # Eq. (2)
    deg = S.sum(1)
    Xt = X - (deg @ X) / deg.sum()                                  # f̃ = f − (fᵀD1 / 1ᵀD1)·1
    fDf = (Xt * (deg[:, None] * Xt)).sum(0)
    fSf = (Xt * (S @ Xt)).sum(0)
    return (fDf - fSf) / fDf.clamp_min(EPS)                         # f̃ᵀLf̃ / f̃ᵀDf̃ with L = D − S

def spec_score(X, S):                                               # Eq. (4), SPEC ϕ1 with identity spectrum
    deg = S.sum(1).clamp_min(EPS); sq = deg.sqrt()
    G = sq[:, None] * X
    G = G / G.norm(dim=0, keepdim=True).clamp_min(EPS)              # f̂ = D^½ f / ‖D^½ f‖
    Sn = S / (sq[:, None] * sq[None, :])
    fLf = 1.0 - (G * (Sn @ G)).sum(0)                               # f̂ᵀ L_n f̂,  L_n = I − D^-½ S D^-½
    xi1 = sq / sq.norm()                                            # trivial eigenvector of L_n
    return fLf / (1.0 - (xi1 @ G) ** 2).clamp_min(1e-6)

def correlation_filter(X, order, tau, d_max):
    Z = X[:, order]; Z = (Z - Z.mean(0)) / Z.std(0).clamp_min(EPS)
    C = ((Z.T @ Z) / len(Z)).abs().cpu()
    kept = []
    for i in range(len(order)):
        if not kept or C[i, kept].max() <= tau:
            kept.append(i)
            if len(kept) >= d_max: break
    return order[torch.tensor(kept, device=order.device)]

def select_channels(P, P_pert, D_full, cfg, use_var, ranking, pair_graph, corr_filter, rng):
    # P: (N,D) normal patches, P_pert: (N,D) patches of T(x) at identical locations (or None)
    cand = torch.arange(P.shape[1], device=P.device)
    if use_var:
        cand = variance_filter(P, cfg["var_quantile"])
    if ranking is None:
        return cand, None
    d = min(len(cand), max(1, int(round(cfg["keep_frac"] * D_full))))
    N = len(P)
    if pair_graph:
        n = min(N, cfg["n_graph"] // 2)
        sub = torch.from_numpy(rng.choice(N, n, replace=False)).to(P.device)
        X = torch.cat([P[sub][:, cand], P_pert[sub][:, cand]]).float()
        S = knn_heat_graph(X, cfg["knn"])
        ar = torch.arange(n, device=P.device)
        S[ar, ar + n] += cfg["lam_pair"]; S[ar + n, ar] += cfg["lam_pair"]     # S = S_knn + λ S_pair
    else:
        n = min(N, cfg["n_graph"])
        sub = torch.from_numpy(rng.choice(N, n, replace=False)).to(P.device)
        X = P[sub][:, cand].float()
        S = knn_heat_graph(X, cfg["knn"])
    scores = laplacian_score(X, S) if ranking == "ls" else spec_score(X, S)
    order = torch.argsort(scores)                                   # ascending: lower = better
    kept = correlation_filter(X, order, cfg["corr_thresh"], d) if corr_filter else order[:d]
    return cand[kept], scores[kept]

## 7. Memory bank: greedy coreset with sparse random projection, and nearest-neighbour search

In [ ]:
def sparse_random_projection(dim_in, dim_out, device, seed=0):
    g = torch.Generator().manual_seed(seed)                         # Achlioptas: ±√(3/k) w.p. 1/6, 0 w.p. 2/3
    r = torch.rand(dim_in, dim_out, generator=g)
    R = torch.zeros(dim_in, dim_out); R[r < 1/6] = 1.0; R[r > 5/6] = -1.0
    return (R * math.sqrt(3.0 / dim_out)).to(device)

def greedy_coreset(X, n_select, proj_dim, seed=0):
    N = len(X)
    if n_select >= N: return torch.arange(N, device=X.device)
    Z = X @ sparse_random_projection(X.shape[1], proj_dim, X.device, seed) if X.shape[1] > proj_dim else X
    sel = torch.empty(n_select, dtype=torch.long, device=X.device)
    sel[0] = int(np.random.default_rng(seed).integers(N))
    min_d = (Z - Z[sel[0]]).pow(2).sum(1)
    for i in range(1, n_select):
        sel[i] = torch.argmax(min_d)
        min_d = torch.minimum(min_d, (Z - Z[sel[i]]).pow(2).sum(1))
    return sel

@torch.no_grad()
def nn_search(Q, M, bs=2048):
    dists, idxs = [], []
    for i in range(0, len(Q), bs):
        d = torch.cdist(Q[i:i + bs], M)
        v, j = d.min(1); dists.append(v); idxs.append(j)
    return torch.cat(dists), torch.cat(idxs)                        # Eq. (3): A(p) = min_m ‖φ_J(p) − m‖

## 8. The detector — one class, all three modes (Section III-A)
Flags switch components on/off, which is exactly what the ablation in Table II needs:

| flag | meaning |
|---|---|
| `use_var` | variance filter |
| `ranking` | `None` (keep all), `"ls"` Laplacian score, `"spec"` SPEC |
| `pair_graph` | consistency graph with photometric pair edges (illumination mode) |
| `corr_filter` | correlation redundancy filter |
| `multiscale` | separate banks for stage 2 and stage 3 instead of one concatenated bank |
| `fusion` | `"mean"` or `"adaptive"` (softmax over mean SPEC score, Eq. 5) |

No network weights are updated; `fit` only estimates channel masks, memory banks and normalisation constants.

In [ ]:
def gaussian_smooth(maps, sigma):
    if sigma <= 0: return maps
    k = int(2 * round(4 * sigma) + 1)
    return TF.gaussian_blur(maps[:, None], kernel_size=k, sigma=sigma)[:, 0]

class SelectiveCore:
    def __init__(self, cfg, use_var=False, ranking=None, pair_graph=False, corr_filter=False,
                 multiscale=False, fusion="adaptive"):
        self.cfg = cfg; self.use_var = use_var; self.ranking = ranking
        self.pair_graph = pair_graph; self.corr_filter = corr_filter
        self.multiscale = multiscale; self.fusion = fusion

    def _levels(self, feats):
        if self.multiscale: return {"layer2": feats["layer2"].float(), "layer3": feats["layer3"].float()}
        return {"concat": concat_levels(feats)}

    @torch.no_grad()
    def fit(self, fit_feats, pert_feats=None, seed=0):
        cfg, rng = self.cfg, np.random.default_rng(seed)
        lv, lvp = self._levels(fit_feats), (self._levels(pert_feats) if pert_feats is not None else None)
        self.banks = {}
        for name, fmap in lv.items():
            n_img, D, h, w = fmap.shape
            P = to_patches(fmap).to(DEVICE)
            Pp = to_patches(lvp[name]).to(DEVICE) if (self.pair_graph and lvp is not None) else None
            sel, sc = select_channels(P, Pp, D, cfg, self.use_var, self.ranking,
                                      self.pair_graph and Pp is not None, self.corr_filter, rng)
            P = P[:, sel].contiguous(); del Pp
            N = len(P); perm = torch.from_numpy(rng.permutation(N)).to(DEVICE)
            n_hold = int(cfg["holdout_frac"] * N) if self.multiscale else 0
            hold, src = perm[:n_hold], perm[n_hold:]
            n_bank = min(len(src), max(cfg["min_bank"], int(cfg["coreset_ratio"] * len(src))))
            core = src[greedy_coreset(P[src], n_bank, cfg["proj_dim"], seed)]
            bank = P[core].contiguous()
            norm = nn_search(P[hold], bank)[0].mean().item() if n_hold > 0 else 1.0
            self.banks[name] = dict(sel=sel, bank=bank, origin=core.cpu(), norm=max(norm, EPS), grid=(h, w),
                                    D=D, score=(sc.mean().item() if sc is not None else 0.0))
            del P
        names = list(self.banks)
        if self.multiscale and self.fusion == "adaptive":
            s = torch.tensor([self.banks[n]["score"] for n in names])
            wts = torch.softmax(-s / cfg["temperature"], 0).tolist()    # Eq. (5)
        else:
            wts = [1.0 / len(names)] * len(names)
        self.weights = dict(zip(names, wts))
        return self

    @torch.no_grad()
    def predict(self, feats, bs=16):
        n = len(next(iter(feats.values()))); S = self.cfg["crop"]
        maps = []; t0 = time.time()
        for i in range(0, n, bs):
            lv = self._levels(subset(feats, slice(i, i + bs)))
            fused = 0
            for name, B in self.banks.items():
                fmap = lv[name]; b, _, h, w = fmap.shape
                d, _ = nn_search(to_patches(fmap).to(DEVICE)[:, B["sel"]], B["bank"])
                a = (d / B["norm"]).view(b, 1, h, w)
                fused = fused + self.weights[name] * F.interpolate(a, size=(S, S), mode="bilinear", align_corners=False)[:, 0]
            maps.append(gaussian_smooth(fused, self.cfg["smooth_sigma"]).cpu())
        if DEVICE == "cuda": torch.cuda.synchronize()
        ms = (time.time() - t0) * 1000 / n
        maps = torch.cat(maps).numpy()
        return maps, maps.reshape(n, -1).max(1), ms

    def n_channels(self): return sum(len(B["sel"]) for B in self.banks.values())
    def memory_mb(self):  return sum(B["bank"].numel() * 4 for B in self.banks.values()) / 1e6

    def channel_name(self, level, j):
        if level != "concat": return f"{level}:ch{j}"
        return f"layer2:ch{j}" if j < 512 else f"layer3:ch{j - 512}"

## 9. Metrics (Section IV-C)
Image AUROC, pixel AUROC, and AU-PRO integrated up to FPR = 0.3 and normalised (per-region overlap, Bergmann et al.). The AU-PRO here is computed exactly per region via sorted scores, which is fast enough to run for every configuration.

In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz

def au_pro(masks, maps, fpr_limit=0.3, n_steps=300):
    masks = masks.astype(bool)
    neg = maps[~masks]
    fprs = np.linspace(0, fpr_limit, n_steps)
    thr = np.quantile(neg, 1 - fprs)                       # threshold giving each target FPR
    pro, n_reg = np.zeros(n_steps), 0
    for m, a in zip(masks, maps):
        if not m.any(): continue
        lab, k = ndimage.label(m)
        for r in range(1, k + 1):
            s = np.sort(a[lab == r])
            pro += 1 - np.searchsorted(s, thr, side="left") / len(s)
            n_reg += 1
    if n_reg == 0: return float("nan")
    return float(_trapz(pro / n_reg, fprs) / fpr_limit)

def evaluate(y, scores, masks, maps):
    masks = masks.numpy() if torch.is_tensor(masks) else masks
    out = dict(img_auroc=roc_auc_score(y, scores) if len(set(y)) > 1 else float("nan"))
    m = masks.reshape(-1)
    out["pix_auroc"] = roc_auc_score(m, maps.reshape(-1)) if m.any() else float("nan")
    out["pro"] = au_pro(masks, maps)
    return out

## 10. Methods compared
`FEWSHOT_METHODS` → Table I. `ABLATION` → Table II (each row adds one component to the previous one).

In [ ]:
FEWSHOT_METHODS = {
    "PatchCore":         dict(),
    "Ours (few-shot)":   dict(use_var=True, ranking="ls"),
    "Ours (all modes)":  dict(use_var=True, ranking="spec", pair_graph=True, corr_filter=True, multiscale=True, fusion="adaptive"),
}
ABLATION = {
    "PatchCore":             dict(),
    "+ variance filter":     dict(use_var=True),
    "+ Laplacian score":     dict(use_var=True, ranking="ls"),
    "+ consistency graph":   dict(use_var=True, ranking="ls", pair_graph=True),
    "+ correlation filter":  dict(use_var=True, ranking="ls", pair_graph=True, corr_filter=True),
    "+ SPEC per level":      dict(use_var=True, ranking="spec", pair_graph=True, corr_filter=True, multiscale=True, fusion="mean"),
    "+ adaptive fusion":     dict(use_var=True, ranking="spec", pair_graph=True, corr_filter=True, multiscale=True, fusion="adaptive"),
}

def prepare_category(cat):
    data = load_category(DATA_DIR, cat)
    data["f_train"], _       = extract(data["X_train"])
    data["f_test"], bb_ms    = extract(data["X_test"])
    data["X_shift"]          = perturb_batch(data["X_test"], TEST_T, seed=12345)    # fixed lighting-shifted test set
    data["f_shift"], _       = extract(data["X_shift"])
    data["backbone_ms"]      = bb_ms
    return data

def run_one(data, kwargs, k, seed, test_sets=("clean",)):
    n_train = len(data["X_train"])
    rng = np.random.default_rng(1000 + seed)
    fit_idx = np.sort(rng.choice(n_train, k, replace=False)) if (k is not None and k < n_train) else np.arange(n_train)
    fit_feats = subset(data["f_train"], fit_idx)
    pert = None
    if kwargs.get("pair_graph"):
        pert, _ = extract(perturb_batch(data["X_train"][fit_idx], FIT_T, seed=seed))
    model = SelectiveCore(CFG, **kwargs).fit(fit_feats, pert, seed=seed)
    rows = []
    for ts in test_sets:
        maps, scores, ms = model.predict(data["f_test" if ts == "clean" else "f_shift"])
        r = evaluate(data["y_test"], scores, data["M_test"], maps)
        r.update(test=ts, d=model.n_channels(), bank_mb=model.memory_mb(),
                 ms_per_img=ms + data["backbone_ms"], weights=json.dumps({a: round(b, 3) for a, b in model.weights.items()}))
        rows.append(r)
    return model, fit_idx, rows

## 11. Experiment 1 — Few-shot detection (Table I)
k ∈ {1, 2, 4, 8, 16} normal images, several random draws. Results are appended to `fewshot_raw.csv` after every category, so a disconnected runtime can resume where it stopped.

In [ ]:
FEW_CSV = os.path.join(CFG["results_dir"], "fewshot_raw.csv")
done = set(pd.read_csv(FEW_CSV)["category"]) if os.path.exists(FEW_CSV) else set()

for cat in CFG["run_categories"]:
    if cat in done: print("skip", cat, "(already done)"); continue
    data = prepare_category(cat); rows = []
    for k in tqdm(CFG["k_shots"], desc=cat):
        for seed in CFG["seeds"]:
            for mname, kw in FEWSHOT_METHODS.items():
                _, _, rs = run_one(data, kw, k, seed)
                for r in rs: rows.append(dict(category=cat, method=mname, k=k, seed=seed, **r))
    df = pd.DataFrame(rows)
    df.to_csv(FEW_CSV, mode="a", header=not os.path.exists(FEW_CSV), index=False)
    print(df.groupby(["method", "k"])["img_auroc"].mean().unstack().round(4))
    del data; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
few = pd.read_csv(FEW_CSV)
per_cat = few.groupby(["category", "method", "k"])[["img_auroc", "pix_auroc", "pro"]].mean().reset_index()
table1 = (per_cat.groupby(["method", "k"])["img_auroc"].mean() * 100).unstack().round(1)
table1_std = (few.groupby(["method", "k", "seed"])["img_auroc"].mean().groupby(["method", "k"]).std() * 100).unstack().round(1)
print("TABLE I — Image-level AUROC (%) averaged over categories (std over seeds in second table)")
display(table1); display(table1_std)
print("Pixel AUROC (%)"); display((per_cat.groupby(["method", "k"])["pix_auroc"].mean() * 100).unstack().round(1))
print("AU-PRO (%)");      display((per_cat.groupby(["method", "k"])["pro"].mean() * 100).unstack().round(1))

plt.figure(figsize=(6, 4))
for m in table1.index: plt.plot(table1.columns, table1.loc[m], "o-", label=m)
plt.xscale("log", base=2); plt.xticks(CFG["k_shots"], CFG["k_shots"])
plt.xlabel("k normal training images"); plt.ylabel("Image AUROC (%)"); plt.grid(alpha=.3); plt.legend(); plt.title("Few-shot behaviour")
plt.savefig(os.path.join(CFG["results_dir"], "fewshot_curve.png"), dpi=200, bbox_inches="tight"); plt.show()

> **RegAD and WinCLIP+ rows of Table I:** these methods need their own code bases (RegAD trains a registration network; WinCLIP+ needs CLIP). Take their numbers from the original papers under the same k and cite them, or run their official repositories separately — this notebook does not invent them.

## 12. Experiment 2 — Ablation under lighting shift (Table II)
Fitting images are clean; test images are re-lit with `TEST_T`, whose parameter ranges are deliberately **different** from the fit-time `FIT_T` (controlled variant described in Section IV-A). Clean-test numbers are recorded too, so you can show that robustness does not cost clean accuracy.
For the real benchmark, switch `CFG["dataset"]="mvtec_ad2"` and use its `test_public` split (the shifted set is then still produced, but the real lighting changes are already in the data).

In [ ]:
ABL_CSV = os.path.join(CFG["results_dir"], "ablation_raw.csv")
done = set(pd.read_csv(ABL_CSV)["category"]) if os.path.exists(ABL_CSV) else set()

for cat in CFG["run_categories"]:
    if cat in done: print("skip", cat); continue
    data = prepare_category(cat); rows = []
    for seed in CFG["seeds"]:
        for mname, kw in tqdm(ABLATION.items(), desc=f"{cat} seed {seed}"):
            _, _, rs = run_one(data, kw, CFG["ablation_k"], seed, test_sets=("clean", "shift"))
            for r in rs: rows.append(dict(category=cat, config=mname, seed=seed, **r))
    pd.DataFrame(rows).to_csv(ABL_CSV, mode="a", header=not os.path.exists(ABL_CSV), index=False)
    del data; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
abl = pd.read_csv(ABL_CSV)
order = list(ABLATION)
t2 = (abl.groupby(["test", "config"])[["img_auroc", "pix_auroc", "pro"]].mean() * 100).round(1)
t2 = t2.rename(columns={"img_auroc": "Img. AUROC", "pix_auroc": "Pix. AUROC", "pro": "PRO"})
print("TABLE II — under lighting shift"); display(t2.loc["shift"].reindex(order))
print("Same configurations on clean test images"); display(t2.loc["clean"].reindex(order))
eff = abl[abl.test == "shift"].groupby("config")[["d", "bank_mb", "ms_per_img"]].mean().reindex(order).round(2)
print("Efficiency (Section IV-C): selected channels d, memory bank (MB), inference ms / image"); display(eff)
print("Per-category image AUROC under shift"); display((abl[abl.test == "shift"].groupby(["category", "config"])["img_auroc"].mean() * 100).unstack().reindex(columns=order).round(1))

## 13. Qualitative results — anomaly maps, clean vs. lighting-shifted

In [ ]:
VIS_CAT = CFG["run_categories"][0]
data = prepare_category(VIS_CAT)
base, fit_idx, _ = run_one(data, ABLATION["PatchCore"], CFG["ablation_k"], 0)
ours, _, _       = run_one(data, ABLATION["+ adaptive fusion"], CFG["ablation_k"], 0)
print("level weights of the full model:", ours.weights)

defect_ids = np.where(data["y_test"] == 1)[0][:4]
fig, ax = plt.subplots(len(defect_ids), 6, figsize=(18, 3 * len(defect_ids)))
for r, i in enumerate(defect_ids):
    for c, (name, Xs, fs) in enumerate([("clean", data["X_test"], data["f_test"]), ("shift", data["X_shift"], data["f_shift"])]):
        fi = subset(fs, slice(i, i + 1))
        mb, _, _ = base.predict(fi); mo, _, _ = ours.predict(fi)
        ax[r, 3*c].imshow(Xs[i].permute(1, 2, 0)); ax[r, 3*c].contour(data["M_test"][i], colors="lime", linewidths=1)
        ax[r, 3*c].set_title(f"{name}: {data['types'][i]}")
        ax[r, 3*c+1].imshow(mb[0], cmap="jet"); ax[r, 3*c+1].set_title(f"PatchCore ({name})")
        ax[r, 3*c+2].imshow(mo[0], cmap="jet"); ax[r, 3*c+2].set_title(f"Ours ({name})")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(CFG["results_dir"], f"qualitative_{VIS_CAT}.png"), dpi=150); plt.show()

## 14. Explainability by channel decomposition (Section III-F)
For the most anomalous patch, the squared score splits **exactly** into per-channel terms c_j(p) = (φ_j(p) − m*_j)². The cell verifies the identity numerically, lists the top channels with their stage, draws their individual contribution maps and shows the training patch that the nearest memory entry m* came from.

In [ ]:
@torch.no_grad()
def explain(model, data, test_i, fit_idx, X_src=None, top=8, n_maps=3):
    X_src = data["X_test"] if X_src is None else X_src
    fi = subset(data["f_test"] if X_src is data["X_test"] else data["f_shift"], slice(test_i, test_i + 1))
    lv = model._levels(fi)
    # pick the level with the largest weighted contribution at its most anomalous patch
    best = None
    for name, B in model.banks.items():
        P = to_patches(lv[name]).to(DEVICE)[:, B["sel"]]
        d, j = nn_search(P, B["bank"])
        score = model.weights[name] * d.max().item() / B["norm"]
        if best is None or score > best[0]: best = (score, name, P, d, j)
    _, name, P, d, j = best; B = model.banks[name]; h, w = B["grid"]
    p = int(d.argmax()); m_star = B["bank"][j[p]]
    c = (P[p] - m_star) ** 2                                              # Eq. (6)
    print(f"level={name}  A(p)^2 = {d[p].item()**2:.4f}   Σ_j c_j(p) = {c.sum().item():.4f}   (exact decomposition)")
    order = torch.argsort(c, descending=True)[:top]
    names = [model.channel_name(name, int(B["sel"][o])) for o in order]
    share = (c[order] / c.sum()).cpu().numpy()
    C_all = ((P - B["bank"][j]) ** 2)                                     # per-channel maps for every patch
    # matched training patch
    origin = int(B["origin"][j[p]]); img_local, rem = divmod(origin, h * w); oy, ox = divmod(rem, w)
    stride = CFG["crop"] // h; py, px = divmod(p, w)
    crop = lambda X, y, x: X[:, max(0, y*stride - stride): y*stride + 2*stride, max(0, x*stride - stride): x*stride + 2*stride]
    fig = plt.figure(figsize=(18, 7)); gs = fig.add_gridspec(2, 6)
    a = fig.add_subplot(gs[0, 0]); a.imshow(X_src[test_i].permute(1, 2, 0))
    a.add_patch(plt.Rectangle((px*stride - stride, py*stride - stride), 3*stride, 3*stride, ec="r", fc="none", lw=2)); a.set_title("test image + flagged patch"); a.axis("off")
    a = fig.add_subplot(gs[0, 1]); a.imshow(d.view(h, w).cpu() / B["norm"], cmap="jet"); a.set_title(f"A(p) at {name}"); a.axis("off")
    a = fig.add_subplot(gs[0, 2]); a.imshow(crop(X_src[test_i], py, px).permute(1, 2, 0)); a.set_title("flagged patch"); a.axis("off")
    a = fig.add_subplot(gs[0, 3]); a.imshow(crop(data["X_train"][fit_idx[img_local]], oy, ox).permute(1, 2, 0)); a.set_title("matched normal patch (m*)"); a.axis("off")
    a = fig.add_subplot(gs[0, 4:]); a.barh(names[::-1], share[::-1] * 100); a.set_xlabel("% of A²(p)"); a.set_title(f"top-{top} channels")
    for t in range(n_maps):
        a = fig.add_subplot(gs[1, t]); a.imshow(C_all[:, order[t]].view(h, w).cpu(), cmap="magma"); a.set_title(f"c_j map: {names[t]}"); a.axis("off")
    a = fig.add_subplot(gs[1, n_maps]); a.imshow(data["M_test"][test_i], cmap="gray"); a.set_title("ground truth"); a.axis("off")
    cum = np.cumsum((torch.sort(c, descending=True)[0] / c.sum()).cpu().numpy())
    a = fig.add_subplot(gs[1, n_maps + 1:]); a.plot(np.arange(1, len(cum) + 1), cum * 100); a.set_xscale("log")
    a.set_xlabel("number of channels"); a.set_ylabel("cumulative % of A²"); a.grid(alpha=.3); a.set_title("concentration of the explanation")
    plt.tight_layout(); plt.show()
    return pd.DataFrame(dict(channel=names, share_pct=(share * 100).round(2)))

for i in defect_ids[:2]:
    display(explain(ours, data, int(i), fit_idx))

## 15. Export tables for the paper (LaTeX + CSV)

In [ ]:
def to_latex(df, caption, label):
    return df.to_latex(float_format="%.1f", caption=caption, label=label, escape=True)

tex = []
t1 = table1.reindex(columns=[c for c in [1, 4, 16] if c in table1.columns])
t1.columns = [f"k = {c}" for c in t1.columns]
tex.append(to_latex(t1, "Image-level AUROC (\\%) on MVTec AD with k normal training images", "tab:fewshot"))
tex.append(to_latex(t2.loc["shift"].reindex(order), "Ablation under lighting shift", "tab:ablation"))
tex.append(to_latex(eff, "Efficiency: channels, memory-bank size and inference time", "tab:efficiency"))
open(os.path.join(CFG["results_dir"], "tables.tex"), "w").write("\n\n".join(tex))
json.dump(CFG, open(os.path.join(CFG["results_dir"], "config_used.json"), "w"), indent=2)
print("\n\n".join(tex))
print("Saved to", CFG["results_dir"], ":", os.listdir(CFG["results_dir"]))

## 16. Fill the TBD cells of the paper automatically
Upload your paper `.docx` when asked. The cell fills the **PatchCore** and **Ours (few-shot)** rows of Table I with your own runs (RegAD/WinCLIP+ already hold the published numbers), fills every row of Table II with the lighting-shift results, and downloads `paper_with_results.docx`. Text placeholders (abstract summary, discussion paragraphs) are left for you to write from these numbers.

In [ ]:
import zipfile, re, shutil
from xml.sax.saxutils import escape
TBD_RUN = re.compile(r'<w:r>(?:(?!</w:r>).)*?<w:t[^>]*>TBD</w:t></w:r>', re.S)

def _fill_row(xml, label, values, start=0):
    # find the table row whose first cell is exactly `label`, replace its TBD cells left-to-right
    m = re.search(r'<w:t[^>]*>' + re.escape(escape(label)) + r'</w:t>', xml[start:])
    if not m: return xml, 0
    i = start + m.start(); j = xml.find('</w:tr>', i)
    row, n = xml[i:j], 0
    for v in values:
        if v is None: continue
        row, k = TBD_RUN.subn(f'<w:r><w:t>{v}</w:t></w:r>', row, count=1); n += k
    return xml[:i] + row + xml[j:], n

def fill_paper(path_in, path_out):
    # works on both normal and "Strict Open XML" .docx files (edits word/document.xml directly)
    with zipfile.ZipFile(path_in) as z: xml = z.read("word/document.xml").decode("utf8")
    t1_start = xml.find("Method</w:t>"); t2_start = xml.find("Configuration</w:t>"); filled = 0
    ks = [1, 4, 16]
    for name in ["PatchCore", "Ours (few-shot)"]:
        if name in table1.index:
            vals = [f"{table1.loc[name, k]:.1f}" if k in table1.columns and not np.isnan(table1.loc[name, k]) else None for k in ks]
            xml, n = _fill_row(xml, name, vals, t1_start); filled += n
    t2_start = xml.find("Configuration</w:t>")
    tt = t2.loc["shift"]
    for name in tt.index:
        vals = [f"{tt.loc[name, c]:.1f}" for c in ["Img. AUROC", "Pix. AUROC", "PRO"]]
        xml, n = _fill_row(xml, name, vals, t2_start); filled += n
    with zipfile.ZipFile(path_in) as zin, zipfile.ZipFile(path_out, "w", zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = xml.encode("utf8") if item.filename == "word/document.xml" else zin.read(item.filename)
            zout.writestr(item, data)
    print(f"saved {path_out} | filled {filled} cells | TBD left in document: {xml.count('>TBD<')}")

try:
    from google.colab import files
    up = files.upload(); paper_in = list(up)[0]
    out = os.path.join(CFG["results_dir"], "paper_with_results.docx")
    fill_paper(paper_in, out); files.download(out)
except ImportError:
    print("Not in Colab: call fill_paper('your_paper.docx', 'paper_with_results.docx')")

In [ ]:
# Optional: copy results to Google Drive so they survive the runtime
COPY_TO_DRIVE = False
if COPY_TO_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/defect_results && cp -r {CFG['results_dir']}/* /content/drive/MyDrive/defect_results/